In [ ]:
import numpy as np, itertools, sys
def tz(x): return (x&-x).bit_length()-1
def o1(i,j):
    if i==0 or j==0: return 1
    if i==j: return -1
    t=tz(i^j); ti=tz(i); tj=tz(j); k=max(t,ti,tj)+1
    if t>=ti and t>=tj: s=2*((i>>t)&1)-1
    elif ti>t and ti>=tj:
        b=(j>>ti)&1; jl=j&((1<<ti)-1); s=(-1)**b*(1 if jl==0 else -1)
    else: s=1-2*((i>>tj)&1)
    if bin((i|j)>>k).count("1")%2: s=-s
    return s

# 1. split relation: final-doubling split sign = standard sign * (-1)^(a*b) (top bits), check vs block rules of Thm 4.1
def split(i,j,m):
    h=1<<(m-1)
    if i>=h and j>=h:
        il,jl=i-h,j-h
        if il==jl: return 1
        if il==0: return -1
        if jl==0: return 1
        return o1(il,jl)
    return o1(i,j)
bad=0
for m in range(1,9):
    N=1<<m
    for i in range(N):
        for j in range(N):
            a=(i>>(m-1))&1;b=(j>>(m-1))&1
            if split(i,j,m)!=o1(i,j)*(-1)**(a*b): bad+=1
print("split vs std*(-1)^(ab) mismatches:",bad)

# 2. Lemma 8.13
bad=0
for r in range(1,9):
    for k in range(1,2000):
        if o1(2**r*k,2**r*k+1)!=(-1)**(bin(k).count('1')+1): bad+=1
print("Lemma 8.13 mismatches:",bad)

# 3. Statement 8.6 (2x2 block product -1, top-left never unique minority)
for n in range(2,10):
    N=1<<n
    S=np.array([[o1(i,j) for j in range(N)] for i in range(N)])
    A=S[0::2,0::2];B=S[0::2,1::2];C=S[1::2,0::2];D=S[1::2,1::2]
    prod=A*B*C*D
    cond1=(prod==-1).all()
    # unique minority top-left: A differs from B,C,D which all equal
    mino=((B==C)&(C==D)&(A!=B)).any()
    print("n",n,"prod=-1 all:",cond1,"| TL unique minority exists:",mino)

# 4. alphabet ladder and features, n=10
n=10;N=1<<n
S=np.array([[o1(i,j) for j in range(N)] for i in range(N)])
M=(S==-1).astype(int)+1
levels=[M]
sizes=[len(np.unique(M))]
cur=M
while cur.shape[0]>1:
    h=cur.shape[0]//2
    T=np.stack([cur[0::2,0::2],cur[0::2,1::2],cur[1::2,0::2],cur[1::2,1::2]],-1).reshape(h*h,4)
    u,inv=np.unique(T,axis=0,return_inverse=True)
    cur=(inv.reshape(h,h)+1)
    levels.append(cur); sizes.append(len(u))
print("ladder n=10:",sizes)

def feat(a,b):
    def p(x): return 0 if x==0 else bin((a|b)>>tz(x)).count("1")%2
    def q(x,y): return 0 if x==0 else (y>>tz(x))&1
    return (int(b==0),int(a==b),p(a),p(b),p(a^b),q(a,b),q(b,a),q(a^b,a))
mp={}
conf=0
for k in (3,4,5,6):
    L=levels[k];h=L.shape[0]
    for a in range(h):
        for b in range(h):
            f=feat(a,b)
            s=int(L[a,b])
            if f in mp and mp[f]!=s: conf+=1
            mp.setdefault(f,s)
print("distinct feature vectors:",len(mp),", symbols:",len(set(mp.values())),", conflicts across levels 3-6:",conf)
for m in mp:
    print(m)

split vs std*(-1)^(ab) mismatches: 0
Lemma 8.13 mismatches: 0
n 2 prod=-1 all: True | TL unique minority exists: False
n 3 prod=-1 all: True | TL unique minority exists: False
n 4 prod=-1 all: True | TL unique minority exists: False
n 5 prod=-1 all: True | TL unique minority exists: False
n 6 prod=-1 all: True | TL unique minority exists: False
n 7 prod=-1 all: True | TL unique minority exists: False
n 8 prod=-1 all: True | TL unique minority exists: False
n 9 prod=-1 all: True | TL unique minority exists: False
ladder n=10: [2, 6, 12, 12, 12, 12, 12, 12, 9, 4, 1]
distinct feature vectors: 29 , symbols: 12 , conflicts across levels 3-5: 0
(1, 1, 0, 0, 0, 0, 0, 0)
(0, 0, 0, 1, 1, 0, 0, 0)
(0, 0, 0, 0, 0, 0, 0, 0)
(1, 0, 1, 0, 1, 0, 0, 1)
(0, 1, 1, 1, 0, 1, 1, 0)
(0, 0, 0, 1, 0, 0, 0, 1)
(0, 0, 0, 0, 1, 1, 1, 0)
(0, 0, 1, 0, 1, 0, 0, 1)
(0, 0, 1, 1, 0, 1, 1, 0)
(0, 0, 1, 0, 0, 0, 0, 0)
(0, 0, 1, 0, 0, 1, 0, 0)
(0, 0, 0, 1, 1, 1, 0, 0)
(1, 0, 0, 0, 0, 0, 0, 1)
(0, 0, 0, 0, 1, 1, 1, 1)
(0,